# Big Data Analytics: Project 1
**From social-media records to a marketing decision**

**Business question:** A Pakistani clothing brand posts on Instagram. The marketing team wants to know which content receives attention and what it should publish next.

**Today's approach:** solve a small business problem with ordinary tools (CSV + Colab + Pandas). Then scale the data up and measure what changes.

**Learning outcome:** you can inspect a dataset, perform basic EDA, explain one limitation of your workflow, and justify when a scalable tool might help.

**Rules for this notebook**
1. Run the cells in order.
2. Do not assume a column exists. Check it first (Step 5).
3. The demo dataset is **not** verified as data from a Pakistani brand. Make no claims about Pakistan.
4. Repeated or generated rows (Step 8) are a scaling experiment, not new real-world evidence.
5. Write your own findings. Do not copy the "key insights" from the dataset's GitHub README.

---
## Step 4 · Load the data

Download `social_media_sentiment_raw.csv` from the GitHub repository, then run the next cell. In Colab it will ask you to upload the file. Outside Colab, put the file next to this notebook or in a `data/` folder.

A repository is a starting point, not a finished assignment. We must inspect the data and understand what each column actually means.

In [ ]:
# STEP 4 · Load the data
import os, shutil, re, time, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

CSV_NAME = "social_media_sentiment_raw.csv"   # change this if your file has another name
CSV_PATH = None                               # or set a full path here to skip the search

def find_csv():
    if CSV_PATH:
        return CSV_PATH
    for p in [CSV_NAME, os.path.join("data", CSV_NAME),
              os.path.join("/content", CSV_NAME),
              os.path.join("/content/drive/MyDrive", CSV_NAME)]:
        if os.path.exists(p):
            return p
    try:                                       # inside Google Colab: ask for an upload
        from google.colab import files
        print("Please upload", CSV_NAME)
        uploaded = files.upload()
        return list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"Could not find {CSV_NAME}. Put it next to this notebook, in data/, or set CSV_PATH.")

path = find_csv()
print("Using file:", path)

# Keep a backup copy of the original file. Never edit the original.
backup = path + ".backup"
if not os.path.exists(backup):
    shutil.copy(path, backup)
    print("Backup saved:", backup)

df = pd.read_csv(path)
print("Shape (rows, columns):", df.shape)
display(df.head())
df.info()

In [ ]:
# Do the column names match the GitHub README? (The README is not proof; the file is.)
EXPECTED = ["post_id", "timestamp", "platform", "user", "followers_count", "verified",
            "text", "hashtags", "country", "likes", "retweets_shares", "comments", "sentiment"]
print("Expected columns NOT found in your file:", [c for c in EXPECTED if c not in df.columns])
print("Columns in your file that the README did not list:", [c for c in df.columns if c not in EXPECTED])

def has(frame, *cols):
    """True if every named column exists in the frame."""
    return all(c in frame.columns for c in cols)

If any expected column is missing, or there are extra columns, adapt the later cells to the real names before you continue.

---
## Step 5 · Data objects and attributes

**What is one record?** A post, a comment, a user, or a campaign record? These are not interchangeable. Write your answer here after looking at `df.head()`:

> One row is ______________________.

Then run the checks below.

In [ ]:
# STEP 5 · Inspect structure, missing values, duplicates
print("Rows, columns:", df.shape)
display(df.dtypes)
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

In [ ]:
# Quality checks. These only LOOK at the data; nothing is changed yet.
if has(df, "post_id"):
    print("Duplicate post_id values:", df["post_id"].duplicated().sum())

for c in ["likes", "comments", "retweets_shares"]:
    if has(df, c):
        nums = pd.to_numeric(df[c], errors="coerce")
        print(f"{c}: negative values = {(nums < 0).sum()}, blank or non-numeric = {nums.isna().sum()}")

if has(df, "followers_count"):
    as_text = df["followers_count"].astype("string")
    print("followers_count dtype:", df["followers_count"].dtype,
          "| values containing a comma:", as_text.str.contains(",", regex=False).sum())

if has(df, "timestamp"):
    parsed = pd.to_datetime(df["timestamp"], errors="coerce")
    print("timestamp values that fail to convert:", parsed.isna().sum(), "of", len(df))

for c in ["platform", "sentiment", "country", "verified"]:
    if has(df, c):
        print(f"\n{c}: {df[c].nunique(dropna=False)} distinct raw values")
        display(df[c].value_counts(dropna=False).head(15))

In [ ]:
# Provenance check: does this look like real social-media data?
if has(df, "text"):
    print("Distinct post texts:", df["text"].nunique(), "out of", len(df), "rows")
    display(df["text"].value_counts().head(5))
if has(df, "hashtags"):
    tags = df["hashtags"].dropna().str.findall(r"#\w+").explode()
    print("Distinct hashtags used:", tags.nunique())
if has(df, "user"):
    print("Distinct users:", df["user"].nunique(), "| most posts by one user:", df["user"].value_counts().iloc[0])

**Provenance question for your report:** what do the numbers above suggest about where this data came from? Real posts are rarely this repetitive. Write what you notice in your provenance note. The repository does not say how the data was collected, and it is not verified as data from any Pakistani brand.

**Classify every column** (numerical, categorical, text, date/time, identifier) and decide what each missing value might mean. Do not delete missing values automatically. A blank metric may mean "not collected," not zero.

The next cell builds a data-dictionary skeleton. Fill in the last two columns yourself.

In [ ]:
# Data-dictionary skeleton: you complete "meaning_you_verified" and "type_group"
dictionary = pd.DataFrame({
    "column": df.columns,
    "pandas_type": [str(t) for t in df.dtypes],
    "missing": df.isna().sum().values,
    "distinct_values": df.nunique(dropna=True).values,
    "example": [str(df[c].dropna().iloc[0]) if df[c].notna().any() else "" for c in df.columns],
    "type_group": "",             # numerical / categorical / text / date-time / identifier
    "meaning_you_verified": "",
})
display(dictionary)
dictionary.to_csv("my_data_dictionary.csv", index=False)
print("Saved my_data_dictionary.csv (edit it, or copy the table into your report).")

### Clean a copy (never the original)

Every cleaning choice below is a decision you should be able to defend. Read the printed counts.
- Exact duplicate rows are removed.
- Text categories are trimmed. Platform and sentiment are given consistent capitalization.
- `followers_count` and `timestamp` are converted to proper types.
- Negative counts are impossible, so they become **missing (NaN)**, not zero.
- Missing values are kept, not filled or dropped. Rows with a missing group label (for example a blank country) are left out of group tables, so group counts add up to fewer than the total.

In [ ]:
# CLEAN A COPY
clean = df.copy()

def harmonise(series):
    """Trim spaces and unify capitalisation: every spelling becomes the most common one,
    e.g. 'TWITTER' and 'twitter  ' both become 'Twitter'."""
    s = series.astype("string").str.strip()
    key = s.str.lower()
    canon = s.groupby(key).agg(lambda x: x.value_counts().index[0])
    return key.map(canon)

for c in ["platform", "sentiment", "country", "verified"]:
    if has(clean, c):
        clean[c] = harmonise(clean[c])

if has(clean, "followers_count"):
    clean["followers_count"] = pd.to_numeric(
        clean["followers_count"].astype("string").str.replace(",", "", regex=False).str.strip(),
        errors="coerce")

if has(clean, "timestamp"):
    clean["timestamp"] = pd.to_datetime(clean["timestamp"], errors="coerce")

for c in ["likes", "comments", "retweets_shares"]:
    if has(clean, c):
        clean[c] = pd.to_numeric(clean[c], errors="coerce")
        n_neg = int((clean[c] < 0).sum())
        clean[c] = clean[c].mask(clean[c] < 0)      # impossible value -> unknown, not zero
        print(f"{c}: {n_neg} negative values set to missing")

# Remove duplicates AFTER tidying: some copies differ only in capitalisation or spaces
raw_exact = int(df.duplicated().sum())
n_before = len(clean)
clean = clean.drop_duplicates()
print(f"\nExact duplicate rows in the raw file: {raw_exact}")
print(f"Duplicate rows removed after tidying: {n_before - len(clean)} "
      "(this includes copies that differed only in capitalisation or spacing)")
if has(clean, "post_id"):
    print("Duplicate post_id values still present:", int(clean["post_id"].duplicated().sum()))

# Derived columns (check the examples; the hashtag format in your file may differ)
# A blank hashtags cell most likely means the post had none, so it counts as 0 tags.
# That is a judgement about THIS column. Missing metrics are never treated as zero.
if has(clean, "hashtags"):
    clean["hashtag_count"] = clean["hashtags"].fillna("").astype("string").str.count(r"\w+")

metric_cols = [c for c in ["likes", "comments", "retweets_shares"] if has(clean, c)]
if metric_cols:
    # skipna=False: if any part is missing, the total is missing (a blank is not a zero)
    clean["total_engagement"] = clean[metric_cols].sum(axis=1, skipna=False)
if has(clean, "total_engagement", "followers_count"):
    # per FOLLOWER, not per reach. This dataset has no reach or impressions column.
    clean["engagement_per_1k_followers"] = (
        clean["total_engagement"] / clean["followers_count"].where(clean["followers_count"] > 0) * 1000)

print("\nCleaned shape:", clean.shape)
display(clean.isna().sum())
if has(clean, "platform"):
    display(clean["platform"].value_counts(dropna=False))
if has(clean, "sentiment"):
    display(clean["sentiment"].value_counts(dropna=False))
if has(clean, "country"):
    print("Country values (check for inconsistent spellings):")
    display(clean["country"].value_counts(dropna=False))

In [ ]:
# Focus on the brand's channel: Instagram. Set PLATFORM_FOCUS = None to use every platform.
PLATFORM_FOCUS = "Instagram"

if PLATFORM_FOCUS and has(clean, "platform"):
    work = clean[(clean["platform"] == PLATFORM_FOCUS).fillna(False).astype(bool)].copy()
    print(f"{PLATFORM_FOCUS} rows: {len(work):,} of {len(clean):,}")
    if len(work) == 0:
        print("No rows matched. Check the platform spelling above; using all platforms instead.")
        work = clean.copy()
else:
    work = clean.copy()

focus_label = PLATFORM_FOCUS if len(work) < len(clean) else "all platforms"
print("Analysing:", focus_label, "| rows:", len(work))

---
## Step 6 · EDA: turn columns into useful questions

Choose two or three analyses that **this** dataset can genuinely support.
- This file has a `platform` column but no "content type" column, so platform stands in for content type.
- It has a `timestamp`, so day-of-week questions are allowed only if the timestamps converted cleanly.
- It has **no** reach or impressions, so do not calculate engagement rate on reach.

Questions to answer: Which platforms are most represented? What do the median and mean tell us about typical engagement? Are a few posts dominating the averages? Is there a relationship between likes and comments?

In [ ]:
# 6a · Which platforms are most represented, and what is typical engagement on each?
METRICS = [c for c in ["likes", "comments"] if has(clean, c)]

if has(clean, "platform") and METRICS:
    display(clean["platform"].value_counts())
    # count = posts with a value, mean = average, median = typical post
    display(clean.groupby("platform")[METRICS].agg(["count", "mean", "median"]).round(1))
display(work[METRICS].describe().round(1))

In [ ]:
# 6b · Are a few posts dominating the averages?
for c in METRICS:
    s = work[c].dropna().sort_values(ascending=False)
    if len(s) and s.sum() > 0:
        top_n = max(1, int(len(s) * 0.10))
        share = s.head(top_n).sum() / s.sum()
        print(f"{c}: mean = {s.mean():,.1f}, median = {s.median():,.1f}; "
              f"the top 10% of posts ({top_n}) hold {share:.0%} of all {c}")

In [ ]:
# 6c · Engagement by sentiment. ALWAYS look at the number of posts in each group.
def evidence_table(frame, group_col, metric, min_posts=30):
    t = frame.groupby(group_col)[metric].agg(posts="count", median="median", mean="mean").round(1)
    t["too_few_posts"] = t["posts"] < min_posts
    return t.sort_values("median", ascending=False)

if has(work, "sentiment") and has(work, "comments"):
    display(evidence_table(work, "sentiment", "comments"))
if has(work, "sentiment") and has(work, "likes"):
    display(evidence_table(work, "sentiment", "likes"))

# 6d · Relationship between likes and comments (Spearman is robust to a few huge posts)
if has(work, "likes", "comments"):
    print("Spearman correlation, likes vs comments:",
          round(work[["likes", "comments"]].corr(method="spearman").iloc[0, 1], 3))

In [ ]:
# 6e · Day of week: only valid if timestamps converted cleanly
if has(work, "timestamp", "likes") and work["timestamp"].notna().sum() > 0:
    order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    by_day = (work.assign(day_of_week=work["timestamp"].dt.day_name())
                  .groupby("day_of_week")["likes"].agg(posts="count", median="median", mean="mean")
                  .reindex(order).round(1))
    display(by_day)
    print("Check the 'posts' column before claiming a best day.")
else:
    print("No usable timestamp: make no claims about posting days.")

### Charts

Starter chart from the lecture slides (replace `"sentiment"` with a real categorical column if yours differs):

In [ ]:
# Starter chart from the slides
if has(clean, "sentiment"):
    clean["sentiment"].value_counts().plot(kind="bar")
    plt.title("Record count by sentiment")
    plt.xlabel("Category")
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.show()

Cleaner versions of the charts. They use one calm colour for one series, direct value labels, and a light grid, so the data stands out.

In [ ]:
# Chart style helpers
BLUE, ORANGE = "#2a78d6", "#eb6834"          # blue first, then orange (colour-blind-safe order)
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"

def _style(ax, title, xlabel, ylabel):
    ax.set_facecolor("white")
    ax.set_title(title, loc="left", fontsize=12, color=INK)
    ax.set_xlabel(xlabel, color=INK2)
    ax.set_ylabel(ylabel, color=INK2)
    ax.tick_params(colors=INK2)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

def bar_chart(values, labels, title, xlabel, ylabel, fmt="{:,.0f}"):
    fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
    bars = ax.bar(list(labels), list(values), color=BLUE, edgecolor="white", linewidth=2, width=0.7)
    ax.bar_label(bars, labels=[fmt.format(v) for v in values], color=INK2, padding=3, fontsize=9)
    _style(ax, title, xlabel, ylabel)
    ax.margins(y=0.12)                    # headroom so value labels never touch the title
    if len(labels) > 6:
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    fig.tight_layout()
    plt.show()

In [ ]:
# Chart 1 · Number of posts per platform (all platforms)
if has(clean, "platform"):
    counts = clean["platform"].value_counts()
    bar_chart(counts.values, counts.index, "Posts per platform", "Platform", "Number of posts")

In [ ]:
# Chart 2 · Median comments by sentiment, with the number of posts shown under each label
if has(work, "sentiment", "comments"):
    g = work.groupby("sentiment")["comments"].agg(["count", "median"]).sort_values("median", ascending=False)
    labels = [f"{idx}\n(n={int(n)})" for idx, n in zip(g.index, g["count"])]
    bar_chart(g["median"].values, labels, f"Median comments per post by sentiment ({focus_label})",
              "Sentiment (n = posts with a comment count)", "Median comments", fmt="{:,.1f}")

In [ ]:
# Chart 3 · Distribution of likes. Most posts are small; a few are huge.
if has(work, "likes"):
    vals = work["likes"].dropna()
    fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
    ax.hist(vals, bins=30, color=BLUE, edgecolor="white", linewidth=1)
    _style(ax, f"Distribution of likes per post ({focus_label})", "Likes per post", "Number of posts")
    fig.tight_layout()
    plt.show()

In [ ]:
# Chart 4 · Likes vs comments
if has(work, "likes", "comments"):
    pair = work[["likes", "comments"]].dropna()
    fig, ax = plt.subplots(figsize=(7, 4.5), facecolor="white")
    ax.scatter(pair["likes"], pair["comments"], s=36, color=BLUE, alpha=0.45,
               edgecolor="white", linewidth=0.5)
    _style(ax, f"Likes vs comments ({focus_label})", "Likes", "Comments")
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    fig.tight_layout()
    plt.show()

---
## Step 7 · Business interpretation

Numbers are not recommendations until we explain them. Do not say a type of content is best just because its total likes are higher. Check how many posts are in each group and whether you have reach, impressions, or audience-size information.

Write **one statement per group member or per group**, using your own results from Step 6:

| Part | Your answer |
|---|---|
| **Finding**: what pattern do you actually see in this dataset? | |
| **Evidence**: which count, statistic, or chart supports it? | |
| **Action**: what should a marketer test next? | |
| **Limitation**: what can't this dataset tell us? | |

Template (this is **not** a finding until you run the analysis):
> "In this dataset, category X has a higher median number of comments than category Y. The brand could test more posts in category X and compare results using a consistent metric."

Checklist before you write it down:
- Did I compare medians as well as means?
- Does each group have enough posts (see `posts` and `too_few_posts`)?
- Am I claiming anything about reach, Pakistan, or paid ads? (The data cannot support that.)

In [ ]:
# Evidence tables for your Step 7 statement. Pick the group column that fits your finding.
for group_col, metric in [("sentiment", "comments"), ("country", "comments"), ("verified", "comments")]:
    if has(work, group_col, metric):
        print(f"\n{metric} by {group_col} ({focus_label})")
        display(evidence_table(work, group_col, metric))

---
## Step 8 · Discover the need: scale the workload and measure

Run the **same simple analysis** on the original data, then on repeated copies. Label repeated data as a scaling experiment, not new real-world evidence.

| Run | Multiplier | Approximate rows (from about 2,260) |
|---|---|---|
| Baseline | 1x | 2,000 |
| Medium | 10x | 20,000 |
| Large | 100x | 200,000 |
| Very large | 1000x | 2 million, only if your machine has the memory |

Record: file size, rows, load time, analysis time, memory use, which operation becomes inconvenient, and whether the data fits in memory.

Pandas may handle millions of rows on some computers. There is **no universal row count** at which it "fails". The need for Spark or distributed systems depends on data size, operation, hardware, and service requirements.

In [ ]:
# STEP 8 · Scaling experiment (repeated copies of the cleaned data)
RUN_1000X = False                     # set True only if you have several GB of free memory
FACTORS = [1, 10, 100] + ([1000] if RUN_1000X else [])
GROUP_COL = "platform"
SCALE_METRICS = [c for c in ["likes", "comments"] if has(clean, c)]

def run_analysis(frame):
    """The same simple analysis at every size. Returns seconds taken."""
    t0 = time.perf_counter()
    frame.groupby(GROUP_COL)[SCALE_METRICS].agg(["count", "mean", "median"])
    frame[SCALE_METRICS].describe()
    frame[SCALE_METRICS].corr(method="pearson")
    frame.sort_values(SCALE_METRICS[0], ascending=False).head(10)
    return time.perf_counter() - t0

rows = []
for factor in FACTORS:
    big = pd.concat([clean] * factor, ignore_index=True)
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, "scaled.csv")
        big.to_csv(p, index=False)
        size_mb = os.path.getsize(p) / 1e6
        t0 = time.perf_counter()
        loaded = pd.read_csv(p)
        load_s = time.perf_counter() - t0
    analysis_s = min(run_analysis(loaded) for _ in range(3))     # best of 3 reduces noise
    mem_mb = loaded.memory_usage(deep=True).sum() / 1e6
    rows.append({"multiplier": f"{factor}x", "rows": len(loaded), "file_mb": round(size_mb, 1),
                 "load_seconds": round(load_s, 3), "analysis_seconds": round(analysis_s, 4),
                 "memory_mb": round(mem_mb, 1)})
    print(f"{factor}x done: {len(loaded):,} rows")
    del big, loaded

scaling = pd.DataFrame(rows)
display(scaling)
scaling.to_csv("my_scaling_results.csv", index=False)

In [ ]:
# Chart · How time grows with rows (both axes use a log scale)
fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
ax.plot(scaling["rows"], scaling["load_seconds"], marker="o", markersize=8, linewidth=2,
        color=BLUE, label="Load the CSV")
ax.plot(scaling["rows"], scaling["analysis_seconds"], marker="o", markersize=8, linewidth=2,
        color=ORANGE, label="Run the analysis")
ax.set_xscale("log")
ax.set_yscale("log")
_style(ax, "Time vs number of rows (log scales)", "Rows (repeated copies of the same data)", "Seconds")
ax.legend(frameon=False, labelcolor=INK2)
fig.tight_layout()
plt.show()

### Reflect (write 3-4 sentences in your report)

1. Which operation became slowest or most inconvenient as the rows grew?
2. Did the data still fit comfortably in memory? How much memory did it use at the largest size?
3. Repeated copies of the same rows are easier for a computer than real, varied data. How might that change the result?
4. What would change if the data were 100 GB, arrived every minute, or had to be shared by a whole team?

**Bridge:** Spark is a tool to investigate when data volume, processing time, or distributed storage requirements justify it. It is not a requirement for a 2,000-row file.

---
## Step 9 · Connect to analytics: classification, clustering, regression

A short conceptual demo is enough for Project 1. Each demo below is small and optional. Ask: **which of these techniques would help answer my marketing question, and what evidence would I need?**

| Technique | Idea | Example here |
|---|---|---|
| Classification | Predict a category after defining a defensible label | High vs low engagement |
| Clustering | Group posts with similar features (scale numeric features first) | Natural groups of posts |
| Regression | Predict a numeric outcome if the data supports it | Likes |

**Avoid data leakage.** Do not use a target metric to create a label and then present the same metric as a meaningful independent prediction. In the classification demo, the label is built from likes + comments + shares, so those columns are **not allowed** as features. Always split into training and test data for predictive work.

In [ ]:
# Shared setup for the demos
try:
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression, LinearRegression
    from sklearn.cluster import KMeans
    from sklearn.metrics import accuracy_score, r2_score, mean_absolute_error
    SKLEARN_OK = True
except ImportError:
    SKLEARN_OK = False
    print("scikit-learn is not installed here. Read the ideas above and skip the demos.")

NEEDED = ["likes", "comments", "retweets_shares", "followers_count", "hashtag_count", "platform", "verified", "sentiment"]
DEMO_OK = SKLEARN_OK and has(clean, *NEEDED)
if SKLEARN_OK and not DEMO_OK:
    print("Some columns needed for the demos are missing:", [c for c in NEEDED if not has(clean, c)])

if DEMO_OK:
    demo = clean.dropna(subset=NEEDED).copy()      # demo only: rows with any missing value are skipped
    demo["verified"] = demo["verified"].astype("string")
    print(f"Rows used for the demos: {len(demo):,} of {len(clean):,} (all platforms, to have enough rows)")

    # Features known BEFORE a post goes live. Likes, comments, shares are deliberately excluded.
    FEATURE_NUM = ["followers_count", "hashtag_count"]
    FEATURE_CAT = ["platform", "verified", "sentiment"]
    X = pd.concat([demo[FEATURE_NUM].astype(float),
                   pd.get_dummies(demo[FEATURE_CAT].astype("string"), drop_first=True).astype(float)], axis=1)
    LEAK = ["likes", "comments", "retweets_shares", "total_engagement", "engagement_per_1k_followers"]
    assert not any(c in X.columns for c in LEAK), "Leakage: a target metric is among the features"
    print("Features:", list(X.columns))

In [ ]:
# 9a · Classification: predict HIGH vs LOW engagement
if DEMO_OK:
    total = demo[["likes", "comments", "retweets_shares"]].sum(axis=1)
    y = (total >= total.median()).astype(int)          # label: at or above the median total engagement
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    baseline = max(y_te.mean(), 1 - y_te.mean())       # always guess the bigger class
    print(f"Test accuracy: {acc:.3f} | baseline (guess the bigger class): {baseline:.3f}")
    print("If accuracy is close to the baseline, these features do not predict engagement well.")

In [ ]:
# 9b · Clustering: group similar posts (numeric features are scaled first)
if DEMO_OK:
    cols = ["followers_count", "likes", "comments", "retweets_shares"]
    Z = StandardScaler().fit_transform(np.log1p(demo[cols].astype(float)))
    km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
    profile = demo[cols].assign(cluster=km.labels_).groupby("cluster").median().round(1)
    profile.insert(0, "posts", pd.Series(km.labels_).value_counts().sort_index())
    display(profile)
    print("Clusters are not business segments until you can describe and name them.")

In [ ]:
# 9c · Regression: predict the number of likes from information known before posting
if DEMO_OK:
    y_reg = np.log1p(demo["likes"].astype(float))      # log scale because likes are heavily skewed
    Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y_reg, test_size=0.25, random_state=42)
    reg = make_pipeline(StandardScaler(), LinearRegression()).fit(Xr_tr, yr_tr)
    pred = np.expm1(reg.predict(Xr_te))
    actual = np.expm1(yr_te)
    mae_model = mean_absolute_error(actual, pred)
    mae_base = mean_absolute_error(actual, np.full(len(actual), np.expm1(yr_tr).median()))
    print(f"R-squared on the test set (log scale): {r2_score(yr_te, reg.predict(Xr_te)):.3f}")
    print(f"Mean absolute error in likes: model {mae_model:,.1f} vs always-guess-the-median {mae_base:,.1f}")
    print("If the model barely beats the median guess, do not present it as a useful prediction.")

**Exit question:** Which of these techniques would help answer your marketing question, and what evidence would you need? Write 2-3 sentences in your report.

---
## Step 10 · Deliverable and next step

### Project 1 submission checklist
- [ ] Dataset source, license/provenance note, and data dictionary (`my_data_dictionary.csv`, completed)
- [ ] Record count, column types, missing-value and duplicate checks
- [ ] At least two summary statistics and two useful charts
- [ ] Two evidence-based observations and one marketing recommendation
- [ ] One limitation and a reason a scalable tool might help at larger scale (use `my_scaling_results.csv`)

### Simple 10-mark rubric
| Criterion | Marks |
|---|---|
| Data understanding and course concepts | 1 |
| Cleaning and quality checks | 1 |
| EDA and statistics | 2 |
| Charts and interpretation | 2 |
| Scaling experiment and tool justification | 2 |
| Analytics concept, recommendation, and presentation | 2 |
| **Total** | **10** |

**Closing question:** *What does the data tell us, what business problem does it reveal, and what should the business do next?*

**Optional class hashtag:** `#UMTDataDetective`. Public posting is optional; you may submit privately instead.

**Next project:** add genuine or clearly labelled sample paid-campaign data (spend, impressions, clicks, conversions) before moving to Meta Ads and Google Ads analysis. Organic engagement data alone cannot calculate ROAS or paid-ad performance.